### Kernel: R

## Pre-processamento para pseudo-valores de incidencia e latencia

In [7]:
# Bibliotecas utilizadas
library(pacman)
p_load(survival, dplyr, tidyr)

seed <- 2003
set.seed(seed)

options(scipen = 999)

In [9]:
# Leitura dos dados imputados
dados_treino <- read.csv("../../../dados/dados-processados/dados_treino.csv", stringsAsFactors = FALSE)
dados_teste <- read.csv("../../../dados/dados-processados/dados_teste.csv", stringsAsFactors = FALSE)

covariaveis_rede <- setdiff(
  intersect(names(dados_treino), names(dados_teste)),
  c("id_paciente", "tempo", "delta_t")
)

prepara_base <- function(df, conjunto_label, covariaveis) {
  base <- df %>%
    select(id_paciente, tempo, delta_t, all_of(covariaveis)) %>%
    mutate(
      id_paciente = as.numeric(id_paciente),
      tempo = as.numeric(tempo),
      delta_t = as.numeric(delta_t),
      conjunto = conjunto_label
    ) %>%
    select(id_paciente, tempo, delta_t, all_of(covariaveis), conjunto) %>%
    filter(tempo > 0, !is.na(delta_t))
  base
}

treino_base <- prepara_base(dados_treino, "treino", covariaveis_rede) %>%
  arrange(id_paciente)
teste_base <- prepara_base(dados_teste, "teste", covariaveis_rede) %>%
  arrange(id_paciente)

cancer_mama <- bind_rows(treino_base, teste_base) %>%
  arrange(id_paciente)

head(cancer_mama)

,id_paciente,tempo,delta_t,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,⋯,rhc_diagnostico_e_tratamentos_anteriores,rhc_idade_no_diagnostico_tumor,macroregiao,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,conjunto
,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<int>,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<chr>
1,1,16,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,⋯,Com diag./Sem trat.,73,3,Média Renda,5,2014,8500,12,42,treino
2,2,60,1,Não Branca,0,2,Não,Nunca,Sim,SUS,⋯,Sem diag./Sem trat.,75,3,Média Renda,5,2012,8500,-104,69,treino
3,3,1,1,Não Branca,0,2,Não,Nunca,Nunca,SUS,⋯,Sem diag./Sem trat.,93,1,Alta Renda,5,2018,8500,-41,0,teste
4,4,13,1,Não Branca,0,1,Sim,Nunca,Ex-consumidor,SUS,⋯,Com diag./Sem trat.,29,2,Média Renda,1,2013,8500,19,33,teste
5,5,69,1,Não Branca,1,2,Não,Nunca,Nunca,SUS,⋯,Com diag./Sem trat.,38,1,Baixa Renda,1,2010,8500,83,99,treino
6,6,7,1,Não Branca,1,2,Sim,Nunca,Sim,SUS,⋯,Sem diag./Sem trat.,43,1,Alta Renda,2,2017,min100,-10,46,treino


In [10]:
# Grade de tempos (10 quantis entre 0.05 e 0.95)
grade_tempos <- quantile(
  treino_base$tempo[treino_base$delta_t > 0],
  probs = seq(0.05, 0.95, length.out = 10),
  names = FALSE,
  type = 1
)

grade_tempos

[1]  2  7 11 15 20 27 32 41 53 73


Formula dos pseudo-valores para incidencia e latencia segue o anexo de referencia.

Pseudo-incidencia:
$$\hat{p} = 1 - \hat{S}_{KM}(t_{max})$$
$$\hat{p}^{(-i)} = 1 - \hat{S}_{KM}^{(-i)}(t_{max})$$
$$p_i = n\hat{p} - (n-1)\hat{p}^{(-i)}$$

Pseudo-latencia:
$$\hat{S}^*(t) = \frac{\hat{S}_{KM}(t) - \hat{S}_{KM}(t_{max})}{1 - \hat{S}_{KM}(t_{max})}$$
$$\hat{S}^{*(-i)}(t) = \frac{\hat{S}_{KM}^{(-i)}(t) - \hat{S}_{KM}^{(-i)}(t_{max})}{1 - \hat{S}_{KM}^{(-i)}(t_{max})}$$
$$S^*_i(t) = n\hat{S}^*(t) - (n-1)\hat{S}^{*(-i)}(t)$$

In [ ]:
# Pseudo-valores de incidencia e latencia

calc_pseudo_incidencia <- function(df) {
  km <- survfit(Surv(tempo, delta_t) ~ 1, data = df)
  n <- nrow(df)
  cura <- min(km$surv)

  pseudo_incidencia <- sapply(seq_len(n), function(k) {
    km_k <- update(km, subset = -k)
    cura_k <- min(km_k$surv)
    n * (1 - cura) - (n - 1) * (1 - cura_k)
  })

  list(pseudo = pseudo_incidencia, cura = cura)
}

calc_pseudo_latencia <- function(df, tempos, cura) {
  km <- survfit(Surv(tempo, delta_t) ~ 1, data = df)
  n <- nrow(df)
  s_t <- km$surv[findInterval(tempos, km$time)]

  lat_mat <- sapply(seq_len(n), function(k) {
    km_k <- update(km, subset = -k)
    s_k <- km_k$surv[findInterval(tempos, km_k$time)]
    cura_k <- min(km_k$surv)
    n * (s_t - cura) / (1 - cura) - (n - 1) * (s_k - cura_k) / (1 - cura_k)
  })

  lat_mat <- t(lat_mat)
  colnames(lat_mat) <- paste0("t_", round(tempos, 6))
  lat_mat
}

resultado_incidencia <- calc_pseudo_incidencia(treino_base)
pseudo_incidencia_treino <- resultado_incidencia$pseudo
cura_km <- resultado_incidencia$cura

pseudo_latencia_treino <- calc_pseudo_latencia(
  df = treino_base,
  tempos = grade_tempos,
  cura = cura_km
)

dim(pseudo_latencia_treino)

[1] 4861   10

In [5]:
# Montagem da base de entrada para a rede
n_train <- nrow(treino_base)
m <- length(grade_tempos)

base_expandida_treino <- treino_base[
  rep(seq_len(n_train), each = m),
  c("id_paciente", covariaveis_rede),
  drop = FALSE
]

entrada_rede_treino <- base_expandida_treino %>%
  mutate(
    tempo = rep(grade_tempos, times = n_train),
    pseudo_latencia = as.vector(t(pseudo_latencia_treino)),
    pseudo_incidencia = rep(pseudo_incidencia_treino, each = m),
    conjunto = "treino"
  )

n_test <- nrow(teste_base)

base_expandida_teste <- teste_base[
  rep(seq_len(n_test), each = m),
  c("id_paciente", covariaveis_rede),
  drop = FALSE
]

entrada_rede_teste <- base_expandida_teste %>%
  mutate(
    tempo = rep(grade_tempos, times = n_test),
    pseudo_latencia = NA_real_,
    pseudo_incidencia = NA_real_,
    conjunto = "teste"
  )

entrada_rede <- bind_rows(entrada_rede_treino, entrada_rede_teste) %>%
  select(id_paciente, all_of(covariaveis_rede), tempo, pseudo_incidencia, pseudo_latencia, conjunto)

head(entrada_rede, 20)

,id_paciente,rhc_raca_cor,instrucao,est_cong,rhc_historico_familiar_cancer,rhc_historico_alcool,rhc_historico_tabaco,rhc_origem_encamiamento,rhc_estadiamento_clinico,rhc_primeiro_tratamento_recebido_no_hospital_nenhum,⋯,pndr_renda,faixa_etcat,ano,tipo_hcido,diff_data_1consulta,diff_data_tratamento,tempo,pseudo_incidencia,pseudo_latencia,conjunto
,<dbl>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<int>,<int>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1...1,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,2,1.033849,1.2172350,treino
1.1...2,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,7,1.033849,1.5963752,treino
1.2...3,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,11,1.033849,1.9726230,treino
1.3...4,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,15,1.033849,2.3810284,treino
1.4...5,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,20,1.033849,-3.5369021,treino
1.5...6,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,27,1.033849,-3.1078511,treino
1.6...7,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,32,1.033849,-2.6951815,treino
1.7...8,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,41,1.033849,-2.1743766,treino
1.8...9,1,Branca,0,2,Não,Nunca,Nunca,Não SUS,III e IV,Não,⋯,Média Renda,5,2014,8500,12,42,53,1.033849,-1.5675125,treino


In [ ]:
# Escrita dos arquivos de entrada
dir_entrada <- file.path("..", "dados", "entrada")

write.csv(entrada_rede, file.path(dir_entrada, "entrada_rede.csv"), row.names = FALSE)

[1] "cancer_mama_bruto.csv" "entrada_rede.csv"